<a href="https://colab.research.google.com/github/HashiniDivyanjanee/Hashini-Divyanjanee-HPC-Task-2638122/blob/main/Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile PasswordGeneratorToText.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// the cuda crypt function takes in a raw password of ONLY 2 letters (ONLY LOWERCASE)
// and 2 numbers. It outputs a 10-character encrypted password (plus '\0').
char* cudaCrypt(char* rawPassword) {

    static char newPassword[11]; // use static as a local pointer (as in your original code)

    newPassword[0] = rawPassword[0] + 2;
    newPassword[1] = rawPassword[0] - 2;
    newPassword[2] = rawPassword[0] + 1;
    newPassword[3] = rawPassword[1] + 3;
    newPassword[4] = rawPassword[1] - 3;
    newPassword[5] = rawPassword[1] - 1;
    newPassword[6] = rawPassword[2] + 2;
    newPassword[7] = rawPassword[2] - 2;
    newPassword[8] = rawPassword[3] + 4;
    newPassword[9] = rawPassword[3] - 4;
    newPassword[10] = '\0';

    for(int i = 0; i < 10; i++) {
        if(i >= 0 && i < 6) { // checking all lower case letter limits
            if(newPassword[i] > 122) {
                newPassword[i] = (newPassword[i] - 122) + 97;
            } else if(newPassword[i] < 97) {
                newPassword[i] = (97 - newPassword[i]) + 97;
            }
        } else { // checking number section
            if(newPassword[i] > 57) {
                newPassword[i] = (newPassword[i] - 57) + 48;
            } else if(newPassword[i] < 48) {
                newPassword[i] = (48 - newPassword[i]) + 48;
            }
        }
    }
    return newPassword;
}

int main(void) {
    int n;

    printf("Enter number of random passwords to generate: ");
    if (scanf("%d", &n) != 1 || n <= 0) {
        printf("Invalid number.\n");
        return 1;
    }

    // Open output file
    FILE *fp = fopen("passwords.txt", "w");
    if (fp == NULL) {
        perror("Error opening output file");
        return 1;
    }

    // Seed random number generator
    srand((unsigned int) time(NULL));

    // Buffer for raw password (2 letters + 2 digits + '\0')
    char rawPassword[5];
    rawPassword[4] = '\0';

    for (int i = 0; i < n; i++) {
        // Generate 2 random lowercase letters
        rawPassword[0] = 'a' + (rand() % 26);
        rawPassword[1] = 'a' + (rand() % 26);

        // Generate 2 random digits
        rawPassword[2] = '0' + (rand() % 10);
        rawPassword[3] = '0' + (rand() % 10);

        // Encrypt using your existing function
        char *encrypted = cudaCrypt(rawPassword);

        // Write to file: raw and encrypted
        // Format: aa99 -> something
        fprintf(fp, "%s\n", encrypted);
    }

    fclose(fp);

    printf("Generated %d passwords and saved them to passwords.txt\n", n);

    return 0;
}

Writing PasswordGeneratorToText.c


In [2]:
!gcc -O2 -o PasswordGeneratorToText PasswordGeneratorToText.c
!./PasswordGeneratorToText

Enter number of random passwords to generate: 50
Generated 50 passwords and saved them to passwords.txt


In [7]:
%%writefile crack.cu
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cuda_runtime.h>

#define RAW_LEN 4          // 2 letters + 2 digits
#define ENC_LEN 10         // encrypted password length

// check every CUDA call for errors
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                           \
        if (err != cudaSuccess) {                                           \
            fprintf(stderr, "CUDA error at %s:%d - %s\n", __FILE__, __LINE__, \
                    cudaGetErrorString(err));                               \
            exit(1);                                                        \
        }                                                                    \
    } while (0)

//the provided encryption logic, converted to a __device__ function
__device__ void cudaCryptDevice(const char *raw, char *out) {
    out[0] = raw[0] + 2;
    out[1] = raw[0] - 2;
    out[2] = raw[0] + 1;
    out[3] = raw[1] + 3;
    out[4] = raw[1] - 3;
    out[5] = raw[1] - 1;
    out[6] = raw[2] + 2;
    out[7] = raw[2] - 2;
    out[8] = raw[3] + 4;
    out[9] = raw[3] - 4;

    for (int i = 0; i < ENC_LEN; i++) {
        if (i < 6) { // lowercase letter range
            if (out[i] > 122) out[i] = (out[i] - 122) + 97;
            else if (out[i] < 97) out[i] = (97 - out[i]) + 97;
        } else { // digit range
            if (out[i] > 57) out[i] = (out[i] - 57) + 48;
            else if (out[i] < 48) out[i] = (48 - out[i]) + 48;
        }
    }
}

// kernel: one thread cracks one encrypted password
__global__ void crackKernel(const char *encrypted, int numPasswords,
                             char *decrypted, int *found) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= numPasswords) return;   // extra threads in the last block do nothing

    const char *target = encrypted + (size_t)idx * (ENC_LEN + 1);
    char candidate[ENC_LEN];
    char raw[RAW_LEN];

    for (int a = 0; a < 26; a++) {
        raw[0] = 'a' + a;
        for (int b = 0; b < 26; b++) {
            raw[1] = 'a' + b;
            for (int c = 0; c < 10; c++) {
                raw[2] = '0' + c;
                for (int d = 0; d < 10; d++) {
                    raw[3] = '0' + d;

                    cudaCryptDevice(raw, candidate);

                    bool match = true;
                    for (int k = 0; k < ENC_LEN; k++) {
                        if (candidate[k] != target[k]) { match = false; break; }
                    }
                    if (match) {
                        for (int k = 0; k < RAW_LEN; k++)
                            decrypted[(size_t)idx * (RAW_LEN + 1) + k] = raw[k];
                        decrypted[(size_t)idx * (RAW_LEN + 1) + RAW_LEN] = '\0';
                        found[idx] = 1;
                        return;
                    }
                }
            }
        }
    }
    found[idx] = 0;  // no match found for this password
}

int main(int argc, char *argv[]) {
    if (argc != 2) {
        printf("Usage: %s <encrypted_passwords_file>\n", argv[0]);
        return 1;
    }

    // Step A: read file into CPU memory
    FILE *fp = fopen(argv[1], "r");
    if (!fp) { perror("Cannot open input file"); return 1; }

    int capacity = 1024, numPasswords = 0;
    char **lines = (char **)malloc(capacity * sizeof(char *));
    char buffer[64];
    while (fscanf(fp, "%63s", buffer) == 1) {
        if ((int)strlen(buffer) != ENC_LEN) {
            fprintf(stderr, "Skipping invalid line (expected %d chars): %s\n", ENC_LEN, buffer);
            continue;
        }
        if (numPasswords == capacity) {
            capacity *= 2;
            lines = (char **)realloc(lines, capacity * sizeof(char *));
        }
        lines[numPasswords] = strdup(buffer);
        numPasswords++;
    }
    fclose(fp);

    if (numPasswords == 0) {
        printf("No valid encrypted passwords found in file.\n");
        free(lines);
        return 0;
    }
    printf("Read %d encrypted passwords from file.\n", numPasswords);

    // flatten into one contiguous host buffer (stride ENC_LEN+1 for the '\0')
    char *h_encrypted = (char *)malloc((size_t)numPasswords * (ENC_LEN + 1));
    for (int i = 0; i < numPasswords; i++) {
        memcpy(h_encrypted + (size_t)i * (ENC_LEN + 1), lines[i], ENC_LEN);
        h_encrypted[(size_t)i * (ENC_LEN + 1) + ENC_LEN] = '\0';
        free(lines[i]);
    }
    free(lines);

    char *h_decrypted = (char *)malloc((size_t)numPasswords * (RAW_LEN + 1));
    int  *h_found     = (int  *)malloc((size_t)numPasswords * sizeof(int));

    // Step B: allocate GPU memory + copy to GPU
    char *d_encrypted = NULL, *d_decrypted = NULL;
    int  *d_found = NULL;

    CUDA_CHECK(cudaMalloc((void **)&d_encrypted, (size_t)numPasswords * (ENC_LEN + 1)));
    CUDA_CHECK(cudaMalloc((void **)&d_decrypted, (size_t)numPasswords * (RAW_LEN + 1)));
    CUDA_CHECK(cudaMalloc((void **)&d_found,     (size_t)numPasswords * sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_encrypted, h_encrypted,
                           (size_t)numPasswords * (ENC_LEN + 1), cudaMemcpyHostToDevice));

    // dynamic block/thread split based on number of passwords
    int threadsPerBlock = 256;
    if (numPasswords < threadsPerBlock) threadsPerBlock = numPasswords;
    int numBlocks = (numPasswords + threadsPerBlock - 1) / threadsPerBlock;
    printf("Launching %d blocks x %d threads (%d total threads) for %d passwords\n",
           numBlocks, threadsPerBlock, numBlocks * threadsPerBlock, numPasswords);

    // Step C: decrypt via kernel
    crackKernel<<<numBlocks, threadsPerBlock>>>(d_encrypted, numPasswords, d_decrypted, d_found);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    // Step D: copy results back to CPU
    CUDA_CHECK(cudaMemcpy(h_decrypted, d_decrypted,
                           (size_t)numPasswords * (RAW_LEN + 1), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaMemcpy(h_found, d_found,
                           (size_t)numPasswords * sizeof(int), cudaMemcpyDeviceToHost));

    //Step E: write results to file
    FILE *out = fopen("decrypted.txt", "w");
    if (!out) { perror("Cannot create output file"); return 1; }

    // column headers
    fprintf(out, "%-12s | %s\n", "Hash code", "Decrypted");
    fprintf(out, "-------------+------------\n");

    int cracked = 0;
    for (int i = 0; i < numPasswords; i++) {
        const char *enc = h_encrypted + (size_t)i * (ENC_LEN + 1);
        if (h_found[i]) {
            const char *raw = h_decrypted + (size_t)i * (RAW_LEN + 1);
            fprintf(out, "%s -> %s\n", enc, raw);
            cracked++;
        } else {
            fprintf(out, "%s -> NOT FOUND\n", enc);
        }
    }
    fclose(out);
    printf("Cracked %d / %d passwords. Results written to decrypted.txt\n", cracked, numPasswords);

    // Step F: free memory
    CUDA_CHECK(cudaFree(d_encrypted));
    CUDA_CHECK(cudaFree(d_decrypted));
    CUDA_CHECK(cudaFree(d_found));
    free(h_encrypted);
    free(h_decrypted);
    free(h_found);

    return 0;
}

Overwriting crack.cu


In [8]:
!nvcc -O2 -o crack crack.cu

In [9]:
!./crack passwords.txt

Read 50 encrypted passwords from file.
Launching 1 blocks x 50 threads (50 total threads) for 50 passwords
Cracked 50 / 50 passwords. Results written to decrypted.txt


In [10]:
!head -10 decrypted.txt

Hash code    | Decrypted
-------------+------------
oknfbb7334 -> mc58
sorice9523 -> qf77
njmkeg7344 -> lh50
vruddb7323 -> ta57
yuxrln4053 -> wo21
cxbuoq9562 -> zr72
plohbd9591 -> ne75
wsvlfh7371 -> ui53
